# Auditory World Models & Learnable Front-Ends
### Bridging Raw Audio Transformers (Adieu Convolutions) and Self-Supervised Distillation (DINO) with Cochlear Dynamics
#### Notebook by Erick Oduniyi — *Chaos Hearing Project*

---

This notebook disassembles and connects two key papers:
1. **"Audio Transformers: Transformer Architectures for Large Scale Audio Understanding. Adieu Convolutions"** (Verma & Berger, 2021)
2. **"Emerging Properties in Self-Supervised Vision Transformers"** (Caron et al., 2021) — **DINO**

We explore these architectures through **Python, NumPy, and PyTorch**, mapping them directly onto the core themes of **Chaos Hearing**:
- **Physics of Sound:** Learnable time-frequency representations vs. nonlinear Hopf bifurcation cochlear oscillators.
- **Cognition of Hearing:** Auditory scene analysis, cocktail party source segregation, and illusory continuation.
- **Shape Grammar:** The cross-layer isomorphism of **Self-Supervision (Student-Teacher Alignment ↔ Learnable Filterbank ↔ Illusory Restoration)**.

## 1. The Auditory World Model Paradigm

A **World Model** constructs an internal representation of the environment to simulate, predict, and decompose sensory inputs. While visual world models focus on spatial geometry, object boundaries, and occlusion, an **Auditory World Model** must capture the physics of vibration:

- **Superposition:** Sound sources sum linearly in the air; the model must "de-mix" them.
- **Resonance & Timbre:** Objects are defined by their spectral decay profiles and harmonics.
- **Spectral Masking:** Loud sounds mask quiet ones, requiring predictive completion.

### The Learnable Interface: Hopf Oscillators vs. Learnable Front-ends
In `casa/oscillators.py`, we model the cochlea using active, nonlinear dynamical systems (Hopf bifurcations):

$$\frac{dz}{dt} = (\mu + i\omega_0)z - (a + i/3)|z|^2z + f(t)$$

Verma & Berger's **Audio Transformer** approaches the sensory interface from a deep learning perspective. Rather than using fixed Mel-spectrograms or hand-crafted cochlear equations, it learns a **fully connected front-end** directly from raw audio waveforms.

Let's import our libraries and define our environment.

In [ ]:
import numpy as np
import scipy.signal as signal
import matplotlib.pyplot as plt

# Set dark theme for plotting
plt.style.use('dark_background')
plt.rcParams.update({
    'grid.color': '#21262d',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#8b949e',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'text.color': '#c9d1d9',
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#0d1117'
})


## 2. Waveform Patching

Raw audio waveforms at 16 kHz contain 16,000 samples per second. The Audio Transformer processes 1s audio clips by dividing them into **non-overlapping 25ms patches** (400 samples per patch), yielding a sequence of length $T = 40$.

Let's generate a synthetic audio scene — a mix of a harmonic complex and a frequency chirp — and segment it into patches.

In [ ]:
# Generate synthetic 1s audio signal (16 kHz sampling rate)
fs = 16000
duration = 1.0
t = np.linspace(0, duration, int(fs * duration), endpoint=False)

# Source A: Constant harmonic tone (440 Hz + 880 Hz overtone)
source_a = np.sin(2 * np.pi * 440 * t) + 0.5 * np.sin(2 * np.pi * 880 * t)

# Source B: Frequency Chirp (100 Hz to 2000 Hz)
source_b = signal.chirp(t, f0=100, t1=duration, f1=2000, method='linear')

# Combined acoustic scene
mix = 0.5 * source_a + 0.5 * source_b + np.random.normal(0, 0.1, len(t))

# Segment into 25ms patches (400 samples each)
patch_size = 400
num_patches = len(mix) // patch_size
patches = mix.reshape(num_patches, patch_size)

print(f"Original signal shape: {mix.shape}")
print(f"Segmented patches shape: {patches.shape} (T = {num_patches} steps, Patch size = {patch_size} samples)")

# Visualize the mixture and patch boundaries
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))

ax1.plot(t, mix, color='#58a6ff', alpha=0.8, label='Mixture')
ax1.set_title('1-Second Raw Audio Waveform (16,000 samples)', color='#c9d1d9', fontsize=12)
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Amplitude')
ax1.legend(loc='upper right')

# Plot patches and highlight patch #15
ax2.plot(t, mix, color='#58a6ff', alpha=0.3)
for p in range(num_patches):
    ax2.axvline(p * 0.025, color='#30363d', linestyle='--', alpha=0.7)
    
# Highlight Patch 15
ax2.axvspan(15 * 0.025, 16 * 0.025, color='#ffe66d', alpha=0.2, label='Patch #15 (25ms)')
ax2.set_title('Segmented Waveform (40 Non-overlapping 25ms Patches)', color='#c9d1d9', fontsize=12)
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Amplitude')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()


## 3. Fully Connected Front-End & Learned Filterbanks

Instead of utilizing 2D convolutional layers on spectrograms, the Audio Transformer feeds the 400-dimensional patches into a fully connected layer of **2048 neurons**, followed by a projection to **64 neurons** (embedding dimension $E$). 

$$\text{Patch } [400] \xrightarrow{\text{FC } 2048} [2048] \xrightarrow{\text{FC } 64} \text{Embedding } [64]$$

After training, the 2048 weights in the first layer function as a **problem-specific, non-linear, non-constant bandwidth filterbank**. The filters learn to emulate:
- **Sinusoidal Basis Functions** (to detect specific frequencies).
- **Hanning/Hamming Windows** (to prevent spectral leakage).
- **Onset/Offset Detectors** (transient indicators).
- **Energy Envelopes** (integrators).

Let's simulate and visualize these emerged auditory filters.

In [ ]:
# Time scale for a single 25ms patch (400 samples)
t_patch = np.linspace(-0.0125, 0.0125, patch_size)

# Simulate 4 types of learned filters
f_sin = np.sin(2 * np.pi * 800 * t_patch)                                      # Frequency detector
hanning = 0.5 * (1 + np.cos(2 * np.pi * t_patch / 0.025))
f_hann = np.sin(2 * np.pi * 1500 * t_patch) * hanning                           # Windowed tone detector
f_onset = np.where(t_patch >= 0, np.exp(-t_patch / 0.002) * np.sin(2 * np.pi * 3000 * t_patch), 0)  # Onset transient
f_envelope = np.exp(-(t_patch**2) / (2 * 0.004**2))                            # Volume envelope

fig, axs = plt.subplots(2, 2, figsize=(10, 8))

axs[0, 0].plot(t_patch * 1000, f_sin, color='#58a6ff', lw=2)
axs[0, 0].set_title('Filter A: Sinusoidal Basis (Frequency Tuned)', color='#58a6ff')
axs[0, 0].set_xlabel('Time (ms)')
axs[0, 0].set_ylabel('Weight')

axs[0, 1].plot(t_patch * 1000, f_hann, color='#4ecdc4', lw=2)
axs[0, 1].set_title('Filter B: Windowed Tone (Leakage Suppression)', color='#4ecdc4')
axs[0, 1].set_xlabel('Time (ms)')

axs[1, 0].plot(t_patch * 1000, f_onset, color='#ff6b6b', lw=2)
axs[1, 0].set_title('Filter C: Onset Detector (Transient Spike)', color='#ff6b6b')
axs[1, 0].set_xlabel('Time (ms)')
axs[1, 0].set_ylabel('Weight')

axs[1, 1].plot(t_patch * 1000, f_envelope, color='#ffe66d', lw=2)
axs[1, 1].set_title('Filter D: Energy Envelope (Volume Integrator)', color='#ffe66d')
axs[1, 1].set_xlabel('Time (ms)')

plt.suptitle("Emergent Filters in Audio Transformer's Front-End Layer", color='#c9d1d9', fontsize=14)
plt.tight_layout()
plt.show()


## 4. Multi-Scale Wavelet Embeddings

To process embeddings hierarchically, the paper explores a **multi-scale wavelet-inspired decomposition**. 

Instead of decimating the temporal sequence length through pooling, they divide the 64 embedding dimensions into subsets and apply different average pooling windows (sizes 1, 2, 4, 8) along the time axis:

- **32 dimensions:** Scale 1 (untouched, capturing micro-second details).
- **16 dimensions:** Scale 2 (smoothed with window size 2).
- **8 dimensions:** Scale 4 (smoothed with window size 4).
- **8 dimensions:** Scale 8 (smoothed with window size 8, capturing macro-second context).

Let's simulate this wavelet embedding smoothing operation on a random latent matrix.

In [ ]:
# Create a dummy latent embedding matrix: T=40 steps, E=64 dimensions
np.random.seed(0)
x_latent = np.random.randn(40, 64)

# Multi-scale decomposition function
def apply_wavelet_smoothing(latent_matrix):
    T, E = latent_matrix.shape
    out = np.zeros_like(latent_matrix)
    
    # 1. Scale 1: 32 dimensions (untouched)
    out[:, 0:32] = latent_matrix[:, 0:32]
    
    # 2. Scale 2: 16 dimensions (window size 2)
    for t_idx in range(T):
        w_start = max(0, t_idx - 1)
        out[t_idx, 32:48] = np.mean(latent_matrix[w_start:t_idx+1, 32:48], axis=0)
        
    # 3. Scale 4: 8 dimensions (window size 4)
    for t_idx in range(T):
        w_start = max(0, t_idx - 3)
        out[t_idx, 48:56] = np.mean(latent_matrix[w_start:t_idx+1, 48:56], axis=0)
        
    # 4. Scale 8: 8 dimensions (window size 8)
    for t_idx in range(T):
        w_start = max(0, t_idx - 7)
        out[t_idx, 56:64] = np.mean(latent_matrix[w_start:t_idx+1, 56:64], axis=0)
        
    return out

x_smoothed = apply_wavelet_smoothing(x_latent)

# Visualize the effect of multi-scale smoothing
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

im1 = ax1.imshow(x_latent.T, aspect='auto', cmap='coolwarm', origin='lower')
ax1.set_title('Original Latent Embeddings (E=64, T=40)', color='#c9d1d9')
ax1.set_xlabel('Time Step')
ax1.set_ylabel('Embedding Dimension')
fig.colorbar(im1, ax=ax1)

im2 = ax2.imshow(x_smoothed.T, aspect='auto', cmap='coolwarm', origin='lower')
ax2.set_title('Smoothed Wavelet Embeddings', color='#c9d1d9')
ax2.set_xlabel('Time Step')
# Mark the partition lines
ax2.axhline(31.5, color='#ffffff', linestyle='--', alpha=0.7)
ax2.axhline(47.5, color='#ffffff', linestyle='--', alpha=0.7)
ax2.axhline(55.5, color='#ffffff', linestyle='--', alpha=0.7)
ax2.text(2, 15, 'Scale 1 (Untouched)', color='#ffffff', fontsize=9, fontweight='bold')
ax2.text(2, 38, 'Scale 2 (W=2)', color='#ffffff', fontsize=9, fontweight='bold')
ax2.text(2, 51, 'Scale 4 (W=4)', color='#ffffff', fontsize=9, fontweight='bold')
ax2.text(2, 59, 'Scale 8', color='#ffffff', fontsize=9, fontweight='bold')
fig.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.show()


## 5. Self-Supervised Distillation (DINO) for Sound

In **DINO** (Self-Distillation with No Labels), a Student network is trained to predict the outputs of a Teacher network (which holds an EMA of the Student's weights).

$$\mathcal{L} = - \sum_{k} P_{\text{teacher}}(k) \log P_{\text{student}}(k)$$

To prevent representation collapse (where the networks output the same constant value), DINO uses:
1. **Centering:** Subtracted running mean of the teacher's outputs.
2. **Sharpening:** Software scaling using different temperatures (Student $\tau_s = 0.1$, Teacher $\tau_t = 0.04$).

Let's write a NumPy implementation of this self-supervised loss.

In [ ]:
# Softmax helper with temperature scaling
def softmax(x, tau=1.0):
    e = np.exp(x / tau)
    return e / np.sum(e, axis=-1, keepdims=True)

# Conceptual DINO loss simulation
def dino_loss_numpy(student_logits, teacher_logits, center, tau_s=0.1, tau_t=0.04):
    # Centering and sharpening on the teacher representation
    centered_teacher_logits = teacher_logits - center
    p_teacher = softmax(centered_teacher_logits, tau_t)
    
    # Sharpening on the student representation
    log_p_student = np.log(softmax(student_logits, tau_s) + 1e-8)
    
    # Compute Cross Entropy Loss
    loss = -np.sum(p_teacher * log_p_student, axis=-1)
    return np.mean(loss)

# Simulate logits for 4 patches, projecting to 8 dimensions
np.random.seed(42)
s_logits = np.random.randn(4, 8)
t_logits = np.random.randn(4, 8)
running_center = np.mean(t_logits, axis=0, keepdims=True)

loss_val = dino_loss_numpy(s_logits, t_logits, running_center)
print(f"Simulated DINO Loss: {loss_val:.4f}")


## 6. Shape Grammar & Auditory Scene Decomposition

Let's integrate these concepts into the **Chaos Hearing Shape Grammar** as **Worked Example 9**:

| Layer | Component | Production Rule / Operation |
| :--- | :--- | :--- |
| **Backend** | TileGym | Student-Teacher EMA update & Centering/Sharpening collapse prevention |
| **Interface** | Chaos Hearing | Audio Transformer raw 1D waveform patching & learnable filterbank |
| **Frontend** | McDermott Demos | Illusory Continuation (predictive completion of masked tones) |

### The Auditory Object Permanence
When a sound (like speech or a violin note) is masked by a loud transient burst, the brain does not perceive a gap. It uses its internal **Auditory World Model** to predict the missing region.

DINO's student-teacher multi-crop setup mimics this exact predictive mechanism: it forces the model to predict the global, unmasked acoustic representation from local, cropped, or partially masked representations, creating a robust, self-supervised model of **acoustic object permanence**.

In [ ]:
# Visualizing Illusory Continuation (Auditory Object Restoration)
# If a signal has a gap filled with loud noise, it is perceived as continuous.

t_rest = np.linspace(0, 0.6, 6000)
freq = 440
tone = np.sin(2 * np.pi * freq * t_rest)

# Create a gap in the tone (from 0.2s to 0.4s)
gap_tone = tone.copy()
gap_tone[2000:4000] = 0

# Fill the gap with loud noise
masked_tone = gap_tone.copy()
noise = np.random.normal(0, 1.5, 2000)
masked_tone[2000:4000] = noise

# Perceived signal (restored tone inside the noise)
perceived_tone = tone.copy()
perceived_tone[2000:4000] += 0.3 * noise # Brain hears BOTH tone and noise

fig, axs = plt.subplots(3, 1, figsize=(12, 8))

axs[0].plot(t_rest, gap_tone, color='#ff6b6b')
axs[0].set_title('Stimulus A: Tone with Gap (Perceived as discontinuous)', color='#ff6b6b')
axs[0].set_ylabel('Amplitude')

axs[1].plot(t_rest, masked_tone, color='#ffe66d', alpha=0.8)
axs[1].set_title('Stimulus B: Tone with Noise Burst (Masked)', color='#ffe66d')
axs[1].set_ylabel('Amplitude')

axs[2].plot(t_rest, perceived_tone, color='#4ecdc4', alpha=0.8)
axs[2].set_title('Percept: Illusory Continuation (Continuous tone + noise burst)', color='#4ecdc4')
axs[2].set_xlabel('Time (s)')
axs[2].set_ylabel('Amplitude')

plt.suptitle("Illusory Continuation: Auditory Object Permanence", color='#c9d1d9', fontsize=14)
plt.tight_layout()
plt.show()


## 7. Conclusions

- **Adieu Convolutions** demonstrates that Transformers can operate directly on 1D raw waveforms, learning a dynamic auditory front-end that rivals human hand-crafted models.
- **DINO's Student-Teacher distillation** provides the framework for learning representations of objects from partial views, which corresponds directly to the auditory system's ability to restore masked signals (Illusory Continuation).
- Formalizing these relationships as **cross-layer shape grammar rules** allows us to construct a single unified framework spanning hardware tile reductions, cochlear oscillators, and psychological auditory phenomena.

---
### References
1. Verma, P., & Berger, J. (2021). *Audio Transformers: Transformer Architectures for Large Scale Audio Understanding. Adieu Convolutions.* arXiv:2105.00335
2. Caron, M., et al. (2021). *Emerging Properties in Self-Supervised Vision Transformers.* arXiv:2104.14294
3. Eguíluz, V. M., et al. (2000). *Essential Nonlinearities in Hearing.* Physical Review Letters.
4. Cusimano, M., et al. (2024). *Listening with generative models.* Cognition.